# ActiveCampaign → Google Sheets
Extrai campanhas e automações do ActiveCampaign e envia para o Google Sheets.

**Execute cada célula em ordem clicando no botão ▶ ao lado de cada uma.**

In [ ]:
# CÉLULA 1 — Instalar dependências
!pip install gspread google-auth pandas requests -q

In [ ]:
# CÉLULA 2 — Configurações (preencha aqui)
AC_API_URL = "https://bling25662.api-us1.com"
AC_API_KEY = "1dead444b15a84e095897862e38d623b198b0929cbe0a5463b6d988f85aee6541ceef399"

GOOGLE_SHEET_ID = "1F_RIQpIHZ0qrK-yu_lKHbcoh3qpmzBu5dy69RdQ6Pzg"

In [ ]:
# CÉLULA 3 — Autenticar no Google
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
print("Autenticado no Google com sucesso!")

In [ ]:
# CÉLULA 4 — Funções de extração
import requests
import pandas as pd
from datetime import datetime

HEADERS = {"Api-Token": AC_API_KEY}

def fetch_all(endpoint, key):
    results = []
    offset = 0
    limit = 100
    while True:
        url = f"{AC_API_URL}/api/3/{endpoint}?limit={limit}&offset={offset}"
        resp = requests.get(url, headers=HEADERS)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get(key, [])
        results.extend(batch)
        total = int(data.get("meta", {}).get("total", len(results)))
        offset += limit
        if offset >= total:
            break
    return results

def extract_campaigns():
    campaigns = fetch_all("campaigns", "campaigns")
    rows = []
    for c in campaigns:
        rows.append({
            "ID": c.get("id"),
            "Nome": c.get("name"),
            "Tipo": c.get("type"),
            "Status": c.get("status"),
            "Assunto": c.get("subject"),
            "Remetente Nome": c.get("fromname"),
            "Remetente Email": c.get("fromemail"),
            "Total Enviados": c.get("send_amt", 0),
            "Total Abertos": c.get("opens", 0),
            "Total Cliques": c.get("uniquelinkclicks", 0),
            "Total Descadastros": c.get("unsubscribes", 0),
            "Total Bounces": c.get("hardbounces", 0),
            "Taxa Abertura": c.get("opens_rate", 0),
            "Taxa Clique": c.get("clicks_rate", 0),
            "Data Envio": c.get("sdate"),
            "Data Criacao": c.get("cdate"),
            "Atualizado Em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

def extract_automations():
    automations = fetch_all("automations", "automations")
    rows = []
    for a in automations:
        rows.append({
            "ID": a.get("id"),
            "Nome": a.get("name"),
            "Status": a.get("status"),
            "Ativa": "Sim" if a.get("hidden") == "0" else "Nao",
            "Total Contatos": a.get("contactGoalCount", 0),
            "Contatos Ativos": a.get("activeContacts", 0),
            "Contatos Concluidos": a.get("completeCount", 0),
            "Data Criacao": a.get("cdate"),
            "Data Modificacao": a.get("mdate"),
            "Atualizado Em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

print("Funcoes carregadas!")

In [ ]:
# CÉLULA 5 — Extrair dados do ActiveCampaign
print("Extraindo campanhas...")
df_campaigns = extract_campaigns()
print(f"  {len(df_campaigns)} campanhas encontradas")

print("Extraindo automacoes...")
df_automations = extract_automations()
print(f"  {len(df_automations)} automacoes encontradas")

print("\nPreview campanhas:")
df_campaigns.head(3)

In [ ]:
# CÉLULA 6 — Enviar para o Google Sheets
def upload_to_sheets(df, sheet_name):
    spreadsheet = gc.open_by_key(GOOGLE_SHEET_ID)
    try:
        worksheet = spreadsheet.worksheet(sheet_name)
        worksheet.clear()
    except gspread.exceptions.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(title=sheet_name, rows=5000, cols=30)
    df = df.fillna("").astype(str)
    worksheet.update([df.columns.tolist()] + df.values.tolist())
    print(f"  [{sheet_name}] {len(df)} registros enviados!")

upload_to_sheets(df_campaigns, "Campanhas")
upload_to_sheets(df_automations, "Automacoes")
print("\nPronto! Abra sua planilha para ver os dados.")